# Eilenberger Equation: Clean-Limit Quasiclassical Theory

The **Eilenberger equation** is the quasiclassical transport equation for
superconductors in the **clean (ballistic) limit**, where the mean free path
$\ell$ is much larger than the coherence length $\xi$. It occupies the middle
tier in the quasiclassical hierarchy: more general than Usadel (diffusive limit)
but far cheaper than full BdG (microscopic).

## Position in the Theory Hierarchy

$$
\underbrace{\text{BdG}}_{\lambda_F \sim 0.1\,\text{nm}}
\xrightarrow{\text{Gor'kov}}
\underbrace{\text{Eilenberger}}_{\text{ballistic},\ \ell \gg \xi}
\xrightarrow{\text{dirty limit}}
\underbrace{\text{Usadel}}_{\ell \ll \xi}
$$

| Regime | Condition | Equation | Key variable |
|:------:|:---------:|:--------:|:------------:|
| Microscopic | all $\lambda_F$ | BdG | $(u_n, v_n)$ |
| Clean | $\ell \gg \xi$ | **Eilenberger** | $\hat{g}(\hat{k}_F, x)$ |
| Dirty | $\ell \ll \xi$ | Usadel | $\hat{g}(x)$ |

The Eilenberger equation retains the **Fermi-surface angular dependence**
$\hat{k}_F$, which is averaged out in the Usadel limit. This makes it
essential for:
- High-purity single-crystal films
- Ballistic point contacts
- Systems where Fermi-surface anisotropy matters (e.g. MgB$_2$)

## The Eilenberger Equation

For a 1D superconductor/ferromagnet heterostructure, the Eilenberger
equation for the anomalous quasiclassical propagator $f(\omega_n, \hat{k}_F, x)$
is:

$$
\hbar v_{F,x}\, \frac{\partial f}{\partial x}
= -2\bigl(\omega_n + i E_{\text{ex}}\bigr)\, f
  + 2\Delta(x)\, g
$$

with the normalization condition:

$$
g^2 + f \bar{f} = 1
$$

where:
- $v_{F,x} = v_F \cos\theta_{\hat{k}}$ is the Fermi velocity projected onto $\hat{x}$
- $\omega_n = \pi T(2n+1)$ are Matsubara frequencies
- $E_{\text{ex}}$ is the exchange energy in the F layer
- $\Delta(x)$ is the self-consistent pair potential
- $g$ is the normal Green's function, $f$ the anomalous (pair) propagator
- $\bar{f}$ is the time-reversed anomalous function

Compared to Usadel, the key difference is the retention of $v_{F,x}$: the
solution depends on the **direction** of quasiparticle propagation on the
Fermi surface.

## Riccati Parametrization

Direct numerical integration of the Eilenberger equation is unstable because
the equation has both exponentially growing and decaying solutions. The
**Riccati parametrization** (Schopohl & Maki, 1995) cures this by
substituting:

$$
f = \frac{2a}{1 + a\bar{a}}, \qquad
g = \frac{1 - a\bar{a}}{1 + a\bar{a}}
$$

where $a(\omega_n, \hat{k}_F, x)$ is the **Riccati amplitude**. The
normalization $g^2 + f\bar{f} = 1$ is automatically satisfied.

The Eilenberger equation becomes a first-order Riccati ODE:

$$
\hbar v_{F,x}\, \frac{\partial a}{\partial x}
= -2\bigl(\omega_n + iE_{\text{ex}}\bigr)\, a
  + \Delta(x)\bigl(1 - a^2\bigr)
$$

### Why Riccati?

| Property | Direct ($f$, $g$) | Riccati ($a$) |
|:---------|:--------:|:------:|
| Stability | Exponential blowup | Bounded: $|a| \le 1$ |
| Normalization | Must enforce $g^2 + f\bar{f} = 1$ | Automatic |
| Shooting | Needs matching | Stable single-direction sweep |

Physical interpretation: in the S layer, $|a| \to |\Delta|/(\omega_n + \sqrt{\omega_n^2 + |\Delta|^2})$
(the BCS limit). In the N/F layer far from the interface, $a \to 0$.

In [ ]:
# Riccati integration of the Eilenberger equation in an S/F bilayer
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# --- Physical parameters ---
T = 4.0                    # K
omega_1 = np.pi * T       # first Matsubara frequency
Delta_0 = 1.4              # K (BCS gap proxy)
E_ex_K = 50 * 11.6         # 50 meV in K
hbar_vF = 500.0            # nm·K (hbar·v_F / k_B, typical)

# Spatial parameters
d_S = 30.0   # nm
d_F = 15.0   # nm

def riccati_rhs(x, a_re_im, omega_n, E_ex, Delta, hbar_vFx):
    """RHS of the Riccati equation da/dx."""
    a = a_re_im[0] + 1j * a_re_im[1]
    da = (-2 * (omega_n + 1j * E_ex) * a + Delta * (1 - a**2)) / hbar_vFx
    return [da.real, da.imag]

# BCS bulk value at x = -d_S (deep in S)
a_BCS = Delta_0 / (omega_1 + np.sqrt(omega_1**2 + Delta_0**2))

# Integrate from deep S through interface into F
x_span = (-d_S, d_F)
x_eval = np.linspace(-d_S, d_F, 500)

# Layer-dependent parameters
def delta_profile(x):
    return Delta_0 if x < 0 else 0.0

def eex_profile(x):
    return E_ex_K if x >= 0 else 0.0

def rhs_bilayer(x, y):
    a = y[0] + 1j * y[1]
    da = (-2 * (omega_1 + 1j * eex_profile(x)) * a + delta_profile(x) * (1 - a**2)) / hbar_vF
    return [da.real, da.imag]

sol = solve_ivp(rhs_bilayer, x_span, [a_BCS, 0.0], t_eval=x_eval,
                rtol=1e-10, atol=1e-12, method='RK45')

a_complex = sol.y[0] + 1j * sol.y[1]
f_anom = 2 * a_complex / (1 + np.abs(a_complex)**2)
g_norm = (1 - np.abs(a_complex)**2) / (1 + np.abs(a_complex)**2)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# Riccati amplitude
axes[0].plot(sol.t, sol.y[0], 'b-', lw=2, label=r'Re $a$')
axes[0].plot(sol.t, sol.y[1], 'r--', lw=2, label=r'Im $a$')
axes[0].axvline(0, ls=':', color='grey')
axes[0].set_xlabel('$x$ (nm)', fontsize=12)
axes[0].set_ylabel('$a(x)$', fontsize=12)
axes[0].set_title('Riccati amplitude', fontsize=13)
axes[0].legend(fontsize=10); axes[0].grid(True, alpha=0.3)

# Anomalous Green's function
axes[1].plot(sol.t, np.abs(f_anom), 'k-', lw=2)
axes[1].axvline(0, ls=':', color='grey')
axes[1].set_xlabel('$x$ (nm)', fontsize=12)
axes[1].set_ylabel(r'$|f(x)|$', fontsize=12)
axes[1].set_title('Pair amplitude profile', fontsize=13)
axes[1].grid(True, alpha=0.3)

# Normal Green's function
axes[2].plot(sol.t, g_norm, 'g-', lw=2)
axes[2].axvline(0, ls=':', color='grey')
axes[2].set_xlabel('$x$ (nm)', fontsize=12)
axes[2].set_ylabel(r'$g(x)$', fontsize=12)
axes[2].set_title('Normal Green\'s function', fontsize=13)
axes[2].grid(True, alpha=0.3)

for ax in axes:
    ax.text(-d_S/2, ax.get_ylim()[1]*0.9, 'S', fontsize=14, ha='center', color='blue')
    ax.text(d_F/2, ax.get_ylim()[1]*0.9, 'F', fontsize=14, ha='center', color='red')

fig.tight_layout(); plt.show()

### SUPERMag Eilenberger Solver

The `supermag.eilenberger.solve()` function wraps the C++ Riccati integration
with Gauss–Legendre angular averaging into a single call.  Below we reproduce
the pair-amplitude profile from the manual integration above using the
composable API path.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import supermag

# S/F bilayer parameters (same as manual integration above)
Tc0  = 9.2    # K
d_S  = 30.0   # nm
d_F  = 15.0   # nm
xi_S = 38.0   # nm
E_ex = 50.0   # meV

# Composable C++ path: one call replaces the full Riccati + angular averaging
x_api, f_api = supermag.eilenberger.solve(Tc0, d_S, d_F, xi_S, E_ex, n_grid=500)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Left: pair amplitude from API
axes[0].plot(x_api, f_api, 'k-', lw=2, label='supermag.eilenberger')
axes[0].axvline(0, ls=':', color='grey', alpha=0.5)
axes[0].text(-d_S/2, max(f_api)*0.9, 'S', fontsize=14, ha='center', color='blue')
axes[0].text(d_F/2, max(f_api)*0.9, 'F', fontsize=14, ha='center', color='red')
axes[0].set_xlabel('$x$ (nm)', fontsize=13)
axes[0].set_ylabel(r'$|f(x)|$', fontsize=13)
axes[0].set_title('Eilenberger solution (C++ path)', fontsize=14)
axes[0].legend(fontsize=10); axes[0].grid(True, alpha=0.3)

# Right: compare different exchange energies
for eex, col, ls in [(10, '#2ca02c', '--'), (50, '#1f77b4', '-'), (200, '#d62728', ':')]:
    x_e, f_e = supermag.eilenberger.solve(Tc0, d_S, d_F, xi_S, eex, n_grid=400)
    axes[1].plot(x_e, f_e, color=col, ls=ls, lw=2, label=f'$E_{{ex}}$ = {eex} meV')

axes[1].axvline(0, ls=':', color='grey', alpha=0.5)
axes[1].set_xlabel('$x$ (nm)', fontsize=13)
axes[1].set_ylabel(r'$|f(x)|$', fontsize=13)
axes[1].set_title('Exchange-field dependence (API sweep)', fontsize=14)
axes[1].legend(fontsize=10); axes[1].grid(True, alpha=0.3)

fig.tight_layout(); plt.show()

## Fermi-Surface Averaging

Unlike Usadel, the Eilenberger propagator retains the direction $\hat{k}_F$.
Observable quantities require averaging over the Fermi surface:

$$
\langle f \rangle(x) = \int \frac{d\Omega_{\hat{k}}}{4\pi}\, f(\omega_n, \hat{k}_F, x)
$$

In 1D, this reduces to an integral over the angle $\theta$ between
$\hat{k}_F$ and $\hat{x}$:

$$
\langle f \rangle(x) = \frac{1}{2} \int_0^\pi f(\omega_n, \theta, x)\, \sin\theta\, d\theta
$$

For right-movers ($\cos\theta > 0$) and left-movers ($\cos\theta < 0$),
the Riccati equation is integrated in opposite directions.
The two solutions are then combined at each point.

In [ ]:
# Fermi-surface angle dependence of the pair amplitude
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

T = 4.0
omega_1 = np.pi * T
Delta_0 = 1.4
E_ex_K = 50 * 11.6
hbar_vF_base = 500.0
d_S, d_F = 30.0, 15.0

theta_angles = [0.2, 0.5, np.pi/4, np.pi/3, 1.2]  # radians from normal

fig, ax = plt.subplots(figsize=(8, 5))

for theta_k in theta_angles:
    hbar_vFx = hbar_vF_base * np.cos(theta_k)
    if abs(hbar_vFx) < 1e-6:
        continue
    a_BCS = Delta_0 / (omega_1 + np.sqrt(omega_1**2 + Delta_0**2))
    x_eval = np.linspace(-d_S, d_F, 400)

    def rhs(x, y, vFx=hbar_vFx):
        a = y[0] + 1j * y[1]
        Ex = E_ex_K if x >= 0 else 0.0
        D = Delta_0 if x < 0 else 0.0
        da = (-2*(omega_1 + 1j*Ex)*a + D*(1 - a**2)) / vFx
        return [da.real, da.imag]

    sol = solve_ivp(rhs, (-d_S, d_F), [a_BCS, 0.0], t_eval=x_eval,
                    rtol=1e-10, atol=1e-12)
    a_c = sol.y[0] + 1j * sol.y[1]
    f_val = 2 * a_c / (1 + np.abs(a_c)**2)
    ax.plot(sol.t, np.abs(f_val), lw=2,
            label=rf'$\theta_k$ = {np.degrees(theta_k):.0f}°')

ax.axvline(0, ls=':', color='grey')
ax.set_xlabel('$x$ (nm)', fontsize=13)
ax.set_ylabel(r'$|f(\theta_k, x)|$', fontsize=13)
ax.set_title('Pair amplitude for different Fermi-surface angles', fontsize=14)
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## Clean Limit vs. Dirty Limit

The Usadel equation is recovered from Eilenberger by taking the dirty limit
$\ell \ll \xi$. In this limit, impurity scattering isotropizes the
propagators, and the angular dependence drops out:

$$
f(\hat{k}_F, x) \xrightarrow{\ell \ll \xi} f(x) + \hat{k}_F \cdot \delta\mathbf{f}(x)
$$

The isotropic part $f(x)$ satisfies the Usadel equation with diffusion
constant $D = v_F \ell / 3$.

| Property | Eilenberger (clean) | Usadel (dirty) |
|:---------|:-------------------:|:--------------:|
| Mean free path | $\ell \gg \xi$ | $\ell \ll \xi$ |
| Angular dependence | Full $\hat{k}_F$ | Isotropic |
| Coherence length | $\xi_0 = \hbar v_F / (2\pi T_c)$ | $\xi_D = \sqrt{\hbar D / (2\pi T_c)}$ |
| Typical materials | MgB$_2$, epitaxial films | Sputtered Nb, alloys |
| Numerical cost | $\mathcal{O}(N_\theta \times N_x)$ | $\mathcal{O}(N_x)$ |

In [ ]:
# Comparison: clean (Eilenberger) vs dirty (Usadel) pair amplitude decay in F
import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(0.01, 15, 500)  # nm into F layer
E_ex_K = 50 * 11.6
T = 4.0
omega_1 = np.pi * T

# Dirty limit (Usadel): exponential decay with complex kappa
hbar_D_kB = 3.04e5  # nm^2·K
kappa_dirty = np.sqrt(2 * (omega_1 + 1j * E_ex_K) / hbar_D_kB)
f_dirty = np.exp(-kappa_dirty * x)

# Clean limit (Eilenberger): decay length scale = hbar v_F / (2 E_ex)
hbar_vF = 500.0  # nm·K
xi_clean = hbar_vF / (2 * E_ex_K)
f_clean = np.exp(-(1 + 1j) * x / (2 * xi_clean))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(x, np.abs(f_dirty), 'b-', lw=2, label='Dirty (Usadel)')
axes[0].plot(x, np.abs(f_clean), 'r--', lw=2, label='Clean (Eilenberger)')
axes[0].set_xlabel('$x$ (nm)', fontsize=13)
axes[0].set_ylabel(r'$|f(x)|$', fontsize=13)
axes[0].set_title('Pair amplitude decay: clean vs dirty', fontsize=14)
axes[0].legend(fontsize=11); axes[0].grid(True, alpha=0.3)

axes[1].plot(x, f_dirty.real, 'b-', lw=2, label='Dirty Re $f$')
axes[1].plot(x, f_clean.real, 'r--', lw=2, label='Clean Re $f$')
axes[1].plot(x, f_dirty.imag, 'b:', lw=1.5, label='Dirty Im $f$')
axes[1].plot(x, f_clean.imag, 'r:', lw=1.5, label='Clean Im $f$')
axes[1].set_xlabel('$x$ (nm)', fontsize=13)
axes[1].set_ylabel(r'$f(x)$', fontsize=13)
axes[1].set_title('Oscillatory structure comparison', fontsize=14)
axes[1].legend(fontsize=10); axes[1].grid(True, alpha=0.3)

fig.tight_layout(); plt.show()

In [ ]:
# Clean vs Dirty comparison using both supermag solvers
import numpy as np
import matplotlib.pyplot as plt
import supermag

Tc0 = 9.2; d_S = 30.0; d_F = 15.0
xi_S = 38.0; xi_F = 1.0; E_ex = 50.0

x_eil, f_eil = supermag.eilenberger.solve(Tc0, d_S, d_F, xi_S, E_ex, n_grid=400)
x_usa, Delta_usa = supermag.usadel.solve(Tc0, d_S, d_F, xi_S, xi_F, E_ex, n_grid=400)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(x_eil, f_eil / max(f_eil), 'r-', lw=2, label='Eilenberger (clean)')
ax.plot(x_usa, Delta_usa / max(Delta_usa), 'b--', lw=2, label='Usadel (dirty)')
ax.axvline(0, ls=':', color='grey', alpha=0.5)
ax.set_xlabel('$x$ (nm)', fontsize=13)
ax.set_ylabel('Normalised pair amplitude', fontsize=13)
ax.set_title('Clean vs Dirty limit — composable C++ path', fontsize=14)
ax.legend(fontsize=11); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## Numerical Implementation Notes

The C++ implementation in `solvers/eilenberger.cpp` (currently stub) is designed
to use the Riccati parametrization for stability. The key algorithmic steps are:

1. **Spatial discretization**: Uniform grid spanning S and F layers
2. **Angular quadrature**: Gauss–Legendre over $\theta \in [0, \pi]$ for the
   Fermi-surface average
3. **Directional integration**:
   - Right-movers ($\cos\theta > 0$): integrate $a$ from left to right
   - Left-movers ($\cos\theta < 0$): integrate $\bar{a}$ from right to left
4. **Matsubara sum**: Sum over $\omega_n$ for the self-consistency condition
5. **Self-consistency loop**: Iterate $\Delta(x)$ until convergence

The Riccati variable $|a| \le 1$ eliminates the exponential instabilities
that plague direct integration of $f$ and $g$.

## References

1. Eilenberger, G., "Transformation of Gorkov's equation for type II superconductors into transport-like equations," Z. Physik **214**, 195 (1968).
2. Schopohl, N. & Maki, K., "Quasiparticle spectrum around a vortex line in a d-wave superconductor," Phys. Rev. B **52**, 490 (1995).
3. Eschrig, M., "Distribution functions in nonequilibrium theory of superconductivity and Andreev spectroscopy in unconventional superconductors," Phys. Rev. B **61**, 9061 (2000).
4. Belzig, W. et al., "Quasiclassical Green's function approach to mesoscopic superconductivity," Superlattices Microstruct. **25**, 1251 (1999).